<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day12_a_(260324)_%EB%A1%9C%EA%B7%B8%EC%9D%B8%26%ED%9A%8C%EC%9B%90%EA%B0%80%EC%9E%85_async_await_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');
const jwt = require('jsonwebtoken');
const bcrypt = require('bcrypt');

const app = express();
const PORT = 3000;
const JWT_SECRET = 'my_super_secret_key'; // 클라이언트와 서버가 공유하는 비밀 열쇠

app.use(express.json());

// MariaDB 연결 설정
const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

// [1] 페이지 라우팅: templates 폴더 안의 파일들을 읽어옵니다.
app.get('/', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'login.html')));
app.get('/signup', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'signup.html')));
app.get('/main', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'main.html')));
app.get('/income', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'income.html')));
app.get('/saving', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'saving.html')));

// [2] API: 회원가입 (비밀번호 암호화 후 DB 저장)
app.post('/api/signup', async (req, res) => {
    try {
        const { userId, password, userName } = req.body;
        const hashed = await bcrypt.hash(password, 10);
        await pool.execute('INSERT INTO users (user_id, password, user_name) VALUES (?, ?, ?)', [userId, hashed, userName]);
        res.json({ success: true });
    } catch (err) {
        res.status(500).json({ success: false });
    }
});

// [3] API: 로그인 (검증 성공 시 JWT 발급)
app.post('/api/login', async (req, res) => {
    try {
        const { userId, password } = req.body;
        const [rows] = await pool.execute('SELECT * FROM users WHERE user_id = ?', [userId]);
        const user = rows[0];

        if (user && await bcrypt.compare(password, user.password)) {
            // Payload에 유저 정보를 담아 서명함
            const token = jwt.sign({ userId: user.user_id, userName: user.user_name }, JWT_SECRET, { expiresIn: '1h' });
            res.json({ success: true, token });
        } else {
            res.status(401).json({ success: false });
        }
    } catch (err) {
        res.status(500).json({ success: false });
    }
});

// [4] API: JWT 검증 (클라이언트가 보낸 토큰이 진짜인지 확인)
app.get('/api/verify', (req, res) => {
    const authHeader = req.headers['authorization'];
    const token = authHeader && authHeader.split(' ')[1]; // Bearer 문자열 떼어내기

    if (!token) return res.json({ success: false });

    jwt.verify(token, JWT_SECRET, (err, decoded) => {
        if (err) return res.json({ success: false });
        res.json({ success: true, user: decoded }); // 유효하면 해독된 유저 정보 응답
    });
});

// [5] API: 보호된 데이터 조회
app.get('/api/data', async (req, res) => {
    try {
        const [rows] = await pool.execute('SELECT * FROM spending');
        res.json(rows);
    } catch (err) {
        res.status(500).json([]);
    }
});

app.get('/api/income', async (req, res) => {
    try {
        const [rows] = await pool.execute('SELECT * FROM income');
        res.json(rows);
    } catch (err) {
        res.status(500).json([]);
    }
});

app.get('/api/saving', async (req, res) => {
    try {
        const [rows] = await pool.execute('SELECT * FROM saving');
        res.json(rows);
    } catch (err) {
        res.status(500).json([]);
    }
});

app.listen(PORT, () => console.log(`서버 실행 중: http://localhost:${PORT}`));

Overwriting /content/server.js


In [2]:
%%writefile /content/templates/signup.html

<h1>회원가입</h1>
<input type="text" id="uId" placeholder="아이디"><br>
<input type="password" id="uPw" placeholder="비밀번호"><br>
<input type="text" id="uName" placeholder="이름"><br>
<button onclick="save()">가입하기</button>
<script>
    async function save() {
        await fetch('/api/signup', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({userId: uId.value, password: uPw.value, userName: uName.value})
        });
        alert('가입 완료'); location.href = '/';
    }
</script>

Overwriting /content/templates/signup.html


In [3]:
%%writefile /content/templates/login.html

<h1>로그인</h1>

<input type="text" id="uId" placeholder="아이디"><br>
<input type="password" id="uPw" placeholder="비밀번호"><br>

<button onclick="login()">로그인</button>
<button onclick="location.href='/signup'">회원가입</button>

<script>
    async function login() {
        const res = await fetch('/api/login', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({userId: uId.value, password: uPw.value})
        });

        const data = await res.json();

        if(data.success) {
            localStorage.setItem('myToken', data.token);
            location.href = '/main';
        } else {
            alert('로그인 실패');
        }
    }
</script>

Overwriting /content/templates/login.html


In [4]:
%%writefile /content/templates/main.html

<h1>메인 대시보드</h1>
<p id="welcome"></p>
<button onclick="localStorage.removeItem('myToken'); location.href='/';">로그아웃</button>
<hr>
<h3>지출 내역 (DB에서 비동기 로드)</h3>
<button onclick="location.href='/income'">입금 보러가기</button>
<button onclick="location.href='/saving'">저축 보러가기</button>
<ul id="list"></ul>

<script>
    async function init() {
        const token = localStorage.getItem('myToken');
        if(!token) { location.href = '/'; return; }

        // 1. 토큰 검증 요청 (헤더에 토큰 실어서 보냄)
        const authRes = await fetch('/api/verify', {
            headers: {'Authorization': `Bearer ${token}`}
        });
        const authData = await authRes.json();

        if(authData.success) {
            document.getElementById('welcome').innerText = authData.user.userName + "님 반갑습니다.";
            loadData(); // 인증 성공 시에만 데이터 불러오기
        } else {
            alert('인증이 만료되었습니다.'); location.href = '/';
        }
    }

    async function loadData() {
        const res = await fetch('/api/data');
        const data = await res.json();
        const list = document.getElementById('list');
        data.forEach(item => {
            const li = document.createElement('li');
            li.innerText = `${item.date} - ${item.name}: ${item.amount}원`;
            list.appendChild(li);
        });
    }

    init();
</script>

Overwriting /content/templates/main.html


In [5]:
%%writefile /content/templates/income.html

<h1>입금 대시보드</h1>
<p id="welcome"></p>
<button onclick="localStorage.removeItem('myToken'); location.href='/';">로그아웃</button>
<hr>
<h3>입금 내역 (DB에서 비동기 로드)</h3>
<ul id="list"></ul>

<script>
    async function init() {
        const token = localStorage.getItem('myToken');
        if(!token) { location.href = '/'; return; }

        // 1. 토큰 검증 요청 (헤더에 토큰 실어서 보냄)
        const authRes = await fetch('/api/verify', {
            headers: {'Authorization': `Bearer ${token}`}
        });
        const authData = await authRes.json();

        if(authData.success) {
            document.getElementById('welcome').innerText = authData.user.userName + "님 반갑습니다.";
            loadData(); // 인증 성공 시에만 데이터 불러오기
        } else {
            alert('인증이 만료되었습니다.'); location.href = '/';
        }
    }

    async function loadData() {
        const res = await fetch('/api/income');
        const data = await res.json();
        const list = document.getElementById('list');
        data.forEach(item => {
            const li = document.createElement('li');
            li.innerText = `${item.date} - ${item.name}: ${item.amount}원`;
            list.appendChild(li);
        });
    }

    init();
</script>

Overwriting /content/templates/income.html


In [6]:
%%writefile /content/templates/saving.html

<h1>저축 대시보드</h1>
<p id="welcome"></p>
<button onclick="localStorage.removeItem('myToken'); location.href='/';">로그아웃</button>
<hr>
<h3>저축 내역 (DB에서 비동기 로드)</h3>
<ul id="list"></ul>

<script>
    async function init() {
        const token = localStorage.getItem('myToken');
        if(!token) { location.href = '/'; return; }

        // 1. 토큰 검증 요청 (헤더에 토큰 실어서 보냄)
        const authRes = await fetch('/api/verify', {
            headers: {'Authorization': `Bearer ${token}`}
        });
        const authData = await authRes.json();

        if(authData.success) {
            document.getElementById('welcome').innerText = authData.user.userName + "님 반갑습니다.";
            loadData(); // 인증 성공 시에만 데이터 불러오기
        } else {
            alert('인증이 만료되었습니다.'); location.href = '/';
        }
    }

    async function loadData() {
        const res = await fetch('/api/saving');
        const data = await res.json();
        const list = document.getElementById('list');
        data.forEach(item => {
            const li = document.createElement('li');
            li.innerText = `${item.date} - ${item.name}: ${item.amount}원`;
            list.appendChild(li);
        });
    }

    init();
</script>

Writing /content/templates/saving.html


In [7]:
!npm install express mysql2 jsonwebtoken bcrypt

⠙⠹⠸⠼⠴⠦⠧
up to date, audited 94 packages in 1s
⠧
⠧25 packages are looking for funding
⠧  run `npm fund` for details
⠧
found 0 vulnerabilities
⠧

In [8]:
!nohup node server.js > server.log 2>&1 &

In [9]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 1 package in 1s
⠸

In [10]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [13]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log |tail -n 1

https://activated-processors-older-steam.trycloudflare.com
